# PHASE 4 — UNSUPERVISED LEARNING


# Day 25 — Unsupervised Project: Customer Segmentation


## 1. Learning Objectives
By the end of this project, you will be able to:
- Execute an end-to-end Unsupervised Machine Learning pipeline.
- Combine PCA and K-Means to compress and cluster data simultaneously.
- Use the Elbow Method to scientifically determine the optimal number of clusters.
- Profile clusters to create actionable business "Personas".


## 2. Prerequisites
- Phase 4 Concepts (K-Means, Silhouette Score, PCA, StandardScaler).


## 3. The Business Problem
A retail supermarket has collected basic data on 2,000 customers who own membership cards. They have no idea how to market to these people. 
The Chief Marketing Officer (CMO) asks you: *"Can you group these customers into distinct 'personas' so we can run targeted ad campaigns? Also, can you show me a visual map of these groups?"*


## 4. The Dataset
We have 5 features: Age, Annual Income ($), Spending Score (1-100), Family Size, and Distance to Store (miles).
Notice there is no `y` target variable!


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

# 1. Generate underlying personas (we will act like we don't know these exist)
np.random.seed(42)
X_mock, _ = make_blobs(n_samples=2000, centers=4, n_features=5, cluster_std=1.5, random_state=42)

# Give the features realistic scales
df = pd.DataFrame(X_mock, columns=['Age', 'Annual_Income', 'Spending_Score', 'Family_Size', 'Distance_to_Store'])
df['Age'] = np.abs(df['Age'] * 5 + 40).astype(int)              # 20 to 70
df['Annual_Income'] = np.abs(df['Annual_Income'] * 15000 + 60000) # $40k to $120k
df['Spending_Score'] = np.clip(np.abs(df['Spending_Score'] * 10 + 50), 1, 100) # 1 to 100
df['Family_Size'] = np.clip(np.abs(df['Family_Size'] + 3), 1, 6).astype(int)
df['Distance_to_Store'] = np.abs(df['Distance_to_Store'] * 2 + 5)

print('Dataset Shape:', df.shape)
print('\nFirst 5 rows:')
print(df.head())


## 5. Step 1: Preprocessing and PCA
We have 5 features. We cannot visualize 5 dimensions for the CMO. 
We must scale the data, then use PCA to crush the 5 features down to 2 Principal Components.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

# 2. Pipeline for scaling and PCA
prep_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=2, random_state=42))
])

# We transform the data
X_2d = prep_pipe.fit_transform(df)

pca_step = prep_pipe.named_steps['pca']
variance_kept = sum(pca_step.explained_variance_ratio_)
print(f'Variance kept by 2 components: {variance_kept * 100:.1f}%')


## 6. Step 2: The Elbow Method
Now that we have compressed the data into `X_2d`, we need to figure out how many distinct customer groups exist. We will test K=1 through K=10 and plot the Inertia.


In [ ]:
from sklearn.cluster import KMeans

inertias = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X_2d)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, marker='o', linestyle='--')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Tightness)')
plt.title('The Elbow Method for Customer Segments')
plt.xticks(K_range)
plt.show()


> Look at the graph. The massive drops stop clearly at **K = 4**. We will tell the CMO there are 4 distinct customer personas.


## 7. Step 3: Final Clustering and Visualization
Let's train our final K-Means model with $K=4$ on our 2D data, and plot the customer map!


In [ ]:
from sklearn.metrics import silhouette_score

final_kmeans = KMeans(n_clusters=4, random_state=42)
cluster_labels = final_kmeans.fit_predict(X_2d)

print(f'Final Silhouette Score: {silhouette_score(X_2d, cluster_labels):.2f}\n')

plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=cluster_labels, cmap='Set1', alpha=0.6)
plt.scatter(final_kmeans.cluster_centers_[:, 0], final_kmeans.cluster_centers_[:, 1], 
            c='black', s=300, marker='X', label='Centroids')
plt.title('Customer Segments (PCA 2D Projection)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend()
plt.show()


## 8. Step 4: Profiling the Personas
The CMO loves the graph, but asks: *"What do the red dots actually mean? Are they young? Old? Rich? Poor?"*

Because Principal Components are just math equations, we must attach the `cluster_labels` back to the **original raw dataset** to figure out what each group represents!


In [ ]:
df['Persona_ID'] = cluster_labels

# Calculate the average statistics for each Persona
persona_profiles = df.groupby('Persona_ID').mean().round(1)
print(persona_profiles)


## 9. Business Conclusion
By looking at the averages above, we can name our Personas for the marketing team!
*(Note: Your numbers may vary slightly due to randomness, but the distinct splits will remain).*

1. **Persona 0 (The Wealthy Introverts)**: High Income, Small Family Size, Far from store.
2. **Persona 1 (The Suburban Families)**: Average Income, Very Large Family Size, Medium distance.
3. **Persona 2 (The Bargain Hunters)**: Low Income, Low Spending Score, Very close to store.
4. **Persona 3 (The Big Spenders)**: Very High Spending Score, Medium Income.

The CMO can now send bulk discount diaper coupons to Persona 1, and luxury brand advertisements to Persona 0!


## 10. Phase 4 Evaluation
You have just completed Phase 4! You:
1. Mastered Unsupervised Learning (no `y` target).
2. Built **K-Means** clustering models to find hidden groups.
3. Defeated complex geometry using **DBSCAN** density rules.
4. Evaluated models blindly using the **Elbow Method** and **Silhouette Scores**.
5. Defeated the Curse of Dimensionality using **PCA** compression.
6. Combined PCA and K-Means to execute a real-world Marketing Segmentation pipeline!


## 11. Summary of Phase 4
Unsupervised Learning is the "Wild West" of Machine Learning. It is highly experimental. The algorithms do exactly what you tell them mathematically, but it is ultimately up to the human Data Scientist to look at the resulting clusters, calculate their original feature averages, and derive actual business value from them.

Tomorrow, we begin our final week: **Phase 5 (Advanced Workflows)**, where we will learn Cross-Validation, Hyperparameter Tuning, and how to put these models into Production!
